<a href="https://colab.research.google.com/github/jtech-sudo/Gad/blob/main/Diabetetype2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Librairies fondamentales pour la manipulation de données
import pandas as pd
import numpy as np

# Librairies pour la visualisation des données (EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# Librairies de Scikit-learn pour le prétraitement et la modélisation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

# Librairie pour l'interprétabilité du modèle
import shap

In [ ]:
# importation du jeu de données dans notre environnement
from google.colab import files
uploaded = files.upload()

In [ ]:
# Chargement du jeu de données
import pandas as pd
df = pd.read_csv('diabetes.csv')
df.head()
df.info()
df.describe()


In [ ]:
# --- Exploration initiale ---
print("--- 1. Aperçu des données (5 premières lignes) ---")
print(df.head())


In [ ]:
print("\n--- 2. Informations sur les colonnes et types de données ---")
print(df.info())


In [ ]:
print("\n--- 3. Statistiques descriptives ---")
print(df.describe())


In [ ]:
# --- Analyse visuelle des données ---
import matplotlib.pyplot as plt
import seaborn as sns
# 4. Distribution de la variable cible (Outcome)
print("\n--- 4. Distribution de la variable cible ---")
print(df['Outcome'].value_counts())
sns.countplot(x='Outcome', data=df)
plt.title('Distribution des classes (0: Non-Diabétique, 1: Diabétique)')
plt.show()
# Analyse : Le jeu de données est légèrement déséquilibré, il faudra donc faire attention aux métriques.

# 5. Distribution des caractéristiques (histogrammes)
df.hist(figsize=(12, 10))
plt.suptitle('Distribution des caractéristiques')
plt.show()
# Analyse : On voit que des colonnes comme 'Glucose', 'BloodPressure', 'BMI' ont des valeurs à zéro, ce qui n'est pas réaliste. On voit aussi que 'Insulin' a beaucoup de valeurs nulles.





In [ ]:
# 6. Matrice de corrélation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matrice de corrélation des caractéristiques')
plt.show()
# Analyse : On peut voir les relations entre les variables. Par exemple, 'Glucose' et 'BMI' ont une corrélation positive avec 'Outcome'.


In [ ]:
# --- Nettoyage des données —
# Remplacer les 0 par des NaN dans les colonnes où 0 est une valeur manquante
# Librairies de Scikit-learn pour le prétraitement et la modélisation
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
cols_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_replace] = df[cols_to_replace].replace(0, np.nan)



In [ ]:
# --- Prétraitement pour le Machine Learning ---
# Séparation des caractéristiques (X) et de la variable cible (y)
import numpy as np
from sklearn.tree import DecisionTreeClassifier
cols_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_replace] = df[cols_to_replace].replace(0, np.nan)


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
cols_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_replace] = df[cols_to_replace].replace(0, np.nan)
X = df.drop('Outcome', axis=1)
y = df['Outcome']


In [ ]:
# Séparation des données en ensembles d'entraînement et de test
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Taille de l'ensemble d'entraînement : {X_train.shape}")
print(f"Taille de l'ensemble de test : {X_test.shape}")


In [ ]:
# Standardisation des données (mise à l'échelle)
import numpy as np
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Dictionnaire pour stocker les modèles, les paramètres à tester et les meilleurs résultats
import seaborn as sns
from sklearn.linear_model import LogisticRegression

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}
# A verifier

In [ ]:
# Grille de paramètres pour l'optimisation (GridSearch)
import seaborn as sns
from sklearn.linear_model import LogisticRegression
param_grids = {
    'Logistic Regression': {'C': [0.01, 0.1, 1, 10]},
    'Decision Tree': {'max_depth': [3, 5, 7, 9], 'min_samples_split': [2, 5, 10]},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15]}
}

best_models = {}
# A vérifier indentation?

In [ ]:
# Boucle d'entraînement et d'optimisation
for name, model in models.items():
    print(f"\n--- Entraînement et optimisation du modèle : {name} ---")

    # Utilisation de GridSearchCV pour trouver les meilleurs paramètres
    grid_search = GridSearchCV(model, param_grids[name], cv=5, scoring='f1', n_jobs=-1)
    grid_search.fit(X_train_scaled, y_train)

    # Sauvegarde du meilleur modèle
    best_models[name] = grid_search.best_estimator_
    print(f"Meilleurs paramètres pour {name}: {grid_search.best_params_}")

In [ ]:
# Sauvegarde du meilleur modèle
best_models[name] = grid_search.best_estimator_
print(f"Meilleurs paramètres pour {name}: {grid_search.best_params_}")


In [ ]:
# Boucle d'évaluation sur les meilleurs modèles
print("\n--- Évaluation des modèles optimisés sur l'ensemble de test ---")
for name, model in best_models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    print(f"\n--- Résultats pour {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    print(f"AUC: {roc_auc_score(y_test, y_proba):.4f}")


In [ ]:
# Boucle d'évaluation sur les meilleurs modèles
print("\n--- Évaluation des modèles optimisés sur l'ensemble de test ---")
for name, model in best_models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    print(f"\n--- Résultats pour {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    print(f"AUC: {roc_auc_score(y_test, y_proba):.4f}")

    # Affichage de la matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Diabète', 'Diabète'], yticklabels=['Non-Diabète', 'Diabète'])
    plt.title(f'Matrice de Confusion pour {name}')
    plt.ylabel('Vraie Valeur')
    plt.xlabel('Prédiction du Modèle')
    plt.show()


In [ ]:
    # Affichage de la courbe ROC
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc_score(y_test, y_proba):.2f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Modèle Aléatoire')
    plt.title(f'Courbe ROC pour {name}')
    plt.xlabel('Taux de Faux Positifs')
    plt.ylabel('Taux de Vrais Positifs')
    plt.legend()
    plt.show()


In [ ]:
# Sélection du meilleur modèle basé sur les scores F1 ou AUC (par exemple, le Random Forest)
best_model_name = 'Random Forest' # Supposons que c'est le meilleur
best_model = best_models[best_model_name]
print(f"\n--- Interprétation du meilleur modèle : {best_model_name} ---")


In [ ]:
# Création de l'explainer SHAP
import shap
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

In [ ]:
# Plot de l'importance globale des caractéristiques (résumé SHAP)
print("Shape of shap_values:", len(shap_values))
print("Shape of shap_values[0]:", shap_values[0].shape)
print("Shape of shap_values[1]:", shap_values[1].shape)
print("Shape of X_test_scaled:", X_test_scaled.shape)


In [ ]:
# Conclusion : Le code ci-dessus montre l'importance de chaque variable.
# Un professionnel interpréterait cela en disant, par exemple,
# "Les variables les plus influentes pour la prédiction sont le taux de glucose et l'IMC.
# Cela signifie que plus ces valeurs sont élevées, plus le modèle prédit un risque de diabète."
